# 2. Durable Approvals and MetaHarness Operations

This notebook moves from read-only inspection to a governed workspace edit. It demonstrates why a model-visible `approved: bool` is not human-in-the-loop control and how MemoRizz instead pauses on a durable, exact, single-use proposal.

You will exercise four operational paths:

1. propose, approve, and resume an exact write checkpoint;
2. prove that an approval cannot be consumed twice;
3. prove that a changed workspace invalidates an approved envelope; and
4. cancel a pending run before any adapter code executes.

> **No model or network is used.** The notebook initializes a disposable Git repository because Git provides especially useful workspace fingerprints and diffs.

## Why durable approval is different from prompt consent

```mermaid
sequenceDiagram
    participant C as Client / model
    participant M as MetaHarness
    participant S as Durable approval store
    participant H as Human host
    participant A as Agent harness
    C->>M: Start exact write task
    M->>M: Hash task + args + workspace + policy
    M->>S: Store pending proposal and checkpoint
    M-->>C: pending_approval + opaque proposal ID
    H->>S: Approve proposal ID with identity
    H->>M: Resume proposal ID
    M->>M: Recompute and compare workspace fingerprint
    M->>S: Atomically consume single-use proposal
    M->>A: Execute stored checkpoint
    A-->>M: Outcome and events
    M->>M: Verify and persist evidence
```

The host approves data, not prose: exact tool name, canonical argument hash, policy reason, owner, expiry, approver identity, decision timestamp, and the original checkpoint. The model cannot add `confirm=True`, alter arguments after review, or recreate a more permissive call during resume.

## 1. Create a disposable Git workspace and durable stores

The run and approval ledgers are separate SQLite files. In production they may be opened by different processes: a model-facing worker can propose work while an authenticated UI or CLI performs the decision. The checkpoint survives process restarts because it is stored, not held only in Python memory.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path
from pprint import pprint

from memorizz.approval import ApprovalStateError, SQLiteApprovalStore
from memorizz.metaharness import (
    AdapterOutcome,
    AgentHarness,
    HarnessCapabilities,
    HarnessEvent,
    HarnessEventType,
    HarnessPermissions,
    HarnessTask,
    MetaHarness,
    SQLiteHarnessRunStore,
    VerificationSpec,
)

DEMO_ROOT = Path(tempfile.mkdtemp(prefix="memorizz-metaharness-approval-"))
WORKSPACE = DEMO_ROOT / "workspace"
WORKSPACE.mkdir()

def git(*arguments):
    return subprocess.run(
        ["git", "-C", str(WORKSPACE), *arguments],
        check=True,
        capture_output=True,
        text=True,
    )

git("init", "-q")
git("config", "user.email", "tutorial@memorizz.dev")
git("config", "user.name", "MemoRizz Tutorial")
(WORKSPACE / "README.md").write_text("# Approval demo\n", encoding="utf-8")
(WORKSPACE / "verify_artifact.py").write_text(
    "import os\nfrom pathlib import Path\n"
    "name = os.environ.get('EXPECTED_ARTIFACT', 'release-note.txt')\n"
    "assert Path(name).read_text(encoding='utf-8').strip() == 'created by approved checkpoint'\n",
    encoding="utf-8",
)
git("add", "README.md", "verify_artifact.py")
git("commit", "-qm", "tutorial fixture")

print({"workspace": str(WORKSPACE), "git_status": git("status", "--short").stdout})

## 2. Define a write-capable educational adapter

The adapter deliberately contains no approval logic. That is a control-plane responsibility and must be enforced before `run()` is entered. We count invocations so the notebook can prove canceled and invalidated checkpoints never reached adapter code.

The artifact name lives in `task.metadata`. Metadata is not exposed in the operator-facing proposal, but its cryptographic fingerprint is bound into the approved arguments. This supports private adapter configuration without making it mutable after approval.

In [ ]:
class TutorialWriteHarness(AgentHarness):
    name = "tutorial-write"

    def __init__(self):
        self.invocations = 0

    def probe(self):
        return HarnessCapabilities(
            name=self.name,
            available=True,
            command="in-process educational adapter",
            structured_events=True,
            mcp=False,
            metadata={"network_modes": ["none"], "task_tool_policy": True},
        )

    def run(self, task, *, workspace, context_pack, emit, cancel_event):
        self.invocations += 1
        artifact_name = str(task.metadata.get("artifact_name") or "release-note.txt")
        artifact = workspace / artifact_name
        emit(HarnessEvent(task.run_id, HarnessEventType.FILE_CHANGE, {"path": artifact_name}))
        artifact.write_text("created by approved checkpoint\n", encoding="utf-8")
        return AdapterOutcome(
            final_response=f"Created {artifact_name}",
            usage={"steps": 1},
            cost_usd=0.0,
            exit_code=0,
        )

adapter = TutorialWriteHarness()
service = MetaHarness(
    adapters=[adapter],
    run_store=SQLiteHarnessRunStore(DEMO_ROOT / "runs.sqlite3"),
    approval_store=SQLiteApprovalStore(DEMO_ROOT / "approvals.sqlite3"),
    allowed_workspace_roots=[str(WORKSPACE)],
)

## 3. Propose a direct write

`workspace_mode="direct"` makes approval mandatory by default. Notice that the task schema contains no `approved` or `confirm` field. Calling `service.run()` persists a run and proposal, then returns immediately with `pending_approval`; the adapter invocation counter must remain zero.

The verification command is also part of the approved envelope. We allowlist only `EXPECTED_ARTIFACT` into the child process rather than inheriting the entire host environment.

In [ ]:
os.environ["EXPECTED_ARTIFACT"] = "release-note.txt"
verification_command = f'"{sys.executable}" -B verify_artifact.py'
pending = service.run(
    HarnessTask(
        task="Create the reviewed release note artifact.",
        workspace=str(WORKSPACE),
        harness="tutorial-write",
        memory_id="release-engineering",
        user_id="tutorial-user",
        thread_id="release-42",
        permissions=HarnessPermissions(
            workspace_mode="direct",
            network="none",
            mcp_access="none",
            allowed_env=["EXPECTED_ARTIFACT"],
        ),
        verification=VerificationSpec(command=verification_command),
        metadata={"artifact_name": "release-note.txt"},
    )
)

assert pending.status.value == "pending_approval"
assert adapter.invocations == 0
assert not (WORKSPACE / "release-note.txt").exists()
PROPOSAL_ID = pending.checkpoint["proposal_id"]
print({"run_id": pending.run_id, "proposal_id": PROPOSAL_ID, "adapter_invocations": adapter.invocations})

## 4. Inspect the proposal as a host

The operator-facing record exposes the exact execution envelope and audit fields but not the private checkpoint payload. `argument_hash` is calculated from the canonical arguments and tool name. The proposal ID is an opaque handle; it is not consent by itself.

In a real deployment, inspect and decide through an authenticated host surface:

- SDK: `list_approvals()`, `approve()`, `reject()`, `resume_approval()`;
- CLI: `memorizz harness approvals` and `memorizz harness approve ...`;
- UI: the **Agent Harnesses** approval queue; or
- an application endpoint that authenticates and records the human identity.

The model-facing MCP surface can start, observe, and cancel runs but intentionally cannot approve them.

In [ ]:
proposal = service.list_approvals(status="pending")[0]
pprint(
    {
        "proposal_id": proposal["proposal_id"],
        "tool_name": proposal["tool_name"],
        "argument_hash": proposal["argument_hash"],
        "policy_reason": proposal["policy_reason"],
        "owner_id": proposal["owner_id"],
        "expires_at": proposal["expires_at"],
        "arguments": proposal["arguments"],
    }
)

assert proposal["tool_name"] == "metaharness.run"
assert proposal["arguments"]["workspace_fingerprint"]
assert proposal["arguments"]["context_fingerprint"]
assert proposal["arguments"]["metadata_fingerprint"]

## 5. Approve and resume the stored checkpoint

Approval records **who** made the decision and **why**. Resume then checks the current workspace fingerprint, atomically consumes the proposal, executes the stored task, captures the Git diff, and runs host verification. The adapter is not asked to reinterpret the approval.

In [ ]:
approved = service.approve(
    PROPOSAL_ID,
    approver_id="release-owner@example.com",
    reason="The bounded artifact and verification command were reviewed.",
)
assert approved["status"] == "approved"

result = service.resume_approval(PROPOSAL_ID)
pprint(result.to_dict())

assert result.ok is True
assert result.verified is True
assert adapter.invocations == 1
assert "release-note.txt" in (result.workspace_diff or "")
assert (WORKSPACE / "release-note.txt").read_text() == "created by approved checkpoint\n"

### Single-use means single-use

A replay attempt raises `ApprovalStateError` before the adapter runs. This is important even when the first operation was idempotent: approvals authorize one exact execution, not an unlimited capability token.

In [ ]:
try:
    service.resume_approval(PROPOSAL_ID)
except ApprovalStateError as exc:
    print("Replay blocked:", exc)
else:
    raise AssertionError("A consumed proposal must not be reusable")

assert adapter.invocations == 1

## 6. Invalidate approval when the workspace changes

Approving task arguments is insufficient if the code under those arguments can change before execution. MemoRizz stores a workspace snapshot in the checkpoint and compares it at resume time. The next example approves a second artifact, changes a tracked file, and then attempts resume.

```mermaid
flowchart TD
    A[Proposal approved for fingerprint A] --> B{Fingerprint at resume}
    B -- still A --> C[Consume and execute]
    B -- now B --> D[Fail: workspace_changed_after_approval]
    D --> E[Require a fresh proposal and review]
```

In [ ]:
# Commit the first approved result so the next proposal begins from a clean state.
git("add", "release-note.txt")
git("commit", "-qm", "approved release note")

second = service.run(
    HarnessTask(
        task="Create a second reviewed artifact.",
        workspace=str(WORKSPACE),
        harness="tutorial-write",
        permissions=HarnessPermissions(workspace_mode="direct", network="none", mcp_access="none"),
        metadata={"artifact_name": "second-note.txt"},
    )
)
second_proposal_id = second.checkpoint["proposal_id"]
service.approve(second_proposal_id, approver_id="release-owner@example.com")

# Simulate another process changing the reviewed workspace before resume.
(WORKSPACE / "README.md").write_text("# Changed after approval\n", encoding="utf-8")
invalidated = service.resume_approval(second_proposal_id)

assert invalidated.error_code == "workspace_changed_after_approval"
assert invalidated.status.value == "failed"
assert adapter.invocations == 1
assert not (WORKSPACE / "second-note.txt").exists()
pprint(invalidated.to_dict())

## 7. Cancel before approval

Cancellation is durable too. Canceling a pending approval marks the run canceled and rejects the proposal as a host decision. A later resume is impossible. This gives SDK, CLI, UI, and MCP clients one consistent operational state.

In [ ]:
# Restore the tracked fixture so a fresh proposal can be created.
(WORKSPACE / "README.md").write_text("# Approval demo\n", encoding="utf-8")
assert git("status", "--short").stdout == ""

third = service.run(
    HarnessTask(
        task="Create an artifact that will be canceled.",
        workspace=str(WORKSPACE),
        harness="tutorial-write",
        permissions=HarnessPermissions(workspace_mode="direct", network="none", mcp_access="none"),
        metadata={"artifact_name": "canceled-note.txt"},
    )
)
canceled = service.cancel(third.run_id)
pprint(canceled)

assert canceled["status"] == "canceled"
assert adapter.invocations == 1
assert not (WORKSPACE / "canceled-note.txt").exists()
assert service.get_run(third.run_id)["status"] == "canceled"

## 8. The same lifecycle across every form factor

| Intent | SDK | CLI | Local UI | Model-facing MCP |
|---|---|---|---|---|
| Discover adapters | `list_harnesses()` | `harness doctor` | capability cards | `memorizz_list_harnesses` |
| Start bounded run | `run()` / `start()` | `harness run` | launch form | `memorizz_start_harness_run` |
| Inspect evidence | `get_run()` / `events()` | `harness show --events` | run detail timeline | get-run and get-events tools |
| Decide approval | `approve()` / `reject()` | host approval commands | authenticated approval queue | **intentionally unavailable** |
| Resume checkpoint | `resume_approval()` | approval command resumes | host action | **intentionally unavailable** |
| Cancel | `cancel()` | `harness cancel` | cancel action | `memorizz_cancel_harness_run` |

Example host workflow:

```bash
memorizz harness run "Implement the reviewed fix" --workspace "$PWD" --write --json
memorizz harness approvals --status pending
memorizz harness approve PROPOSAL_ID --approver operator@example.com
memorizz harness show RUN_ID --events --json
```

The next notebook connects this lifecycle to a `MemAgent` and the real vendor adapters.

In [ ]:
service.close()
os.environ.pop("EXPECTED_ARTIFACT", None)
shutil.rmtree(DEMO_ROOT, ignore_errors=True)
print("Removed tutorial resources:", DEMO_ROOT)